# Практикум 4. N-граммные модели и сглаживание

В лаборатории 4 вы считали биграммы, двигали ползунки k и d, считали перплексию и генерировали предложения рулеткой. Здесь то же самое в коде:

1. посчитаем униграммы и биграммы и построим оценку максимального правдоподобия (MLE);
2. реализуем add-k (Лаплас) и интерполированное сглаживание Кнезера–Нея с нуля и проверим, что каждое распределение суммируется в 1;
3. посчитаем перплексию тестовых предложений и подберём k и d;
4. сгенерируем предложения с разной температурой;
5. сравним наши числа с `nltk.lm`.

*Практикум к лаборатории 4 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Счётчики и MLE

Корпус тот же, что в лаборатории: 11 коротких предложений, уже в нижнем регистре и без пунктуации. Каждое предложение обрамляем служебными токенами `<s>` и `</s>`. Биграммная модель считает

$$P_{MLE}(w \mid v) = \frac{c(v\,w)}{c(v)},$$

где $c(v\,w)$ — сколько раз пара встретилась, а $c(v)$ — сколько раз $v$ было контекстом. Словарь предсказаний $V$ — все слова плюс `</s>`.

In [ ]:
corpus = [
    "я люблю горячий чай", "мама любит чай", "папа пьёт чай", "мама пьёт кофе", "я люблю кофе",
    "я люблю нью йорк", "мама любит нью йорк", "нью йорк большой город", "папа видел нью йорк",
    "москва большой город", "москва красивый город",
]
CTX = "пьёт"            # контекст из лаборатории
GEN_CTX = "я"           # контекст для графика в разделе 4
PAIR = ("йорк", "чай")  # слово A частое, но только после «нью»; слово B после разных слов
NEW_CTXS = ["я", "мама", "люблю"]    # ни «йорк», ни «чай» после них не встречались

In [ ]:
import math
from collections import Counter
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

BOS, EOS = "<s>", "</s>"

class BigramCounts:
    def __init__(self, sentences):
        sents = [[BOS] + s.split() + [EOS] for s in sentences]
        self.words = []
        for s in sents:
            for w in s[1:-1]:
                if w not in self.words:
                    self.words.append(w)
        self.pred = self.words + [EOS]           # что предсказываем
        self.hist = [BOS] + self.words           # что может быть контекстом
        self.V = len(self.pred)
        self.big = Counter((s[i - 1], s[i]) for s in sents for i in range(1, len(s)))
        self.uni = Counter(w for s in sents for w in s[1:])
        self.N = sum(self.uni.values())
        self.c = Counter()          # c(v)
        self.n1_after = Counter()   # N1+(v •)
        self.n1_before = Counter()  # N1+(• w)
        for (v, w), n in self.big.items():
            self.c[v] += n
            self.n1_after[v] += 1
            self.n1_before[w] += 1
        self.n1_all = len(self.big)  # N1+(• •)
        self.n_sents = len(sents)

def mle(m, w, v):
    return m.big[v, w] / m.c[v]

M = BigramCounts(corpus)
print(f"{M.n_sents} предложений, {M.N} токенов (без <s>), |V| (слова + </s>) = {M.V}, {M.n1_all} разных биграмм")
print(f"заполнено клеток таблицы: {M.n1_all} / {len(M.hist) * M.V} = {M.n1_all / (len(M.hist) * M.V):.1%}")
print("Самые частые слова:", M.uni.most_common(6))

Таблица биграмм, как в лаборатории: строка — предыдущее слово $v$, столбец — следующее $w$. Почти все клетки пустые, и для MLE каждая пустая клетка означает «невозможно».

In [ ]:
import matplotlib.pyplot as plt

table = pd.DataFrame([[M.big[v, w] for w in M.pred] for v in M.hist], index=M.hist, columns=M.pred)
table["Σ"] = table.sum(axis=1)
print(table.to_string())

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(table.drop(columns="Σ").values, cmap="Blues")
ax.set_xticks(range(M.V)); ax.set_xticklabels(M.pred, rotation=90)
ax.set_yticks(range(len(M.hist))); ax.set_yticklabels(M.hist)
for i, v in enumerate(M.hist):
    for j, w in enumerate(M.pred):
        if M.big[v, w]:
            ax.text(j, i, M.big[v, w], ha="center", va="center", fontsize=8,
                    color="white" if M.big[v, w] > 2 else "black")
ax.set_xlabel("w"); ax.set_ylabel("v"); ax.set_title("c(v w)")
plt.tight_layout()
plt.show()

## 2. Сглаживание: add-k и Кнезер–Ней

**Add-k (при k = 1 — Лаплас)** прибавляет k к каждому счётчику:
$$P_{add\text{-}k}(w \mid v) = \frac{c(v\,w) + k}{c(v) + k\,|V|}.$$

**Интерполированный Кнезер–Ней** вычитает из каждого ненулевого счётчика скидку $d$ и отдаёт освободившуюся массу $\lambda(v)$ распределению **продолжения**:
$$P_{KN}(w \mid v) = \frac{\max(c(v\,w) - d,\, 0)}{c(v)} + \lambda(v)\, P_{cont}(w), \quad \lambda(v) = \frac{d\cdot N_{1+}(v\,\bullet)}{c(v)}, \quad P_{cont}(w) = \frac{N_{1+}(\bullet\,w)}{N_{1+}(\bullet\,\bullet)}.$$

$N_{1+}(\bullet\,w)$ — число **разных** слов, после которых встречалось $w$. Проверим для каждого контекста, что сумма вероятностей по всему словарю равна 1.

In [ ]:
def addk(m, w, v, k=1.0):
    return (m.big[v, w] + k) / (m.c[v] + k * m.V)

def p_cont(m, w):
    return m.n1_before[w] / m.n1_all

def lam(m, v, d=0.75):
    return d * m.n1_after[v] / m.c[v]

def kn(m, w, v, d=0.75):
    return max(m.big[v, w] - d, 0) / m.c[v] + lam(m, v, d) * p_cont(m, w)

METHODS = {"MLE": mle, "add-1": addk, "add-0.1": lambda m, w, v: addk(m, w, v, 0.1), "KN d=0.75": kn}

def check_sums(m):
    worst = 0.0
    for v in m.hist:
        for f in METHODS.values():
            worst = max(worst, abs(sum(f(m, w, v) for w in m.pred) - 1))
    return worst

worst = check_sums(M)
print(f"максимальное отклонение суммы от 1 по всем контекстам и методам: {worst:.1e}")
assert worst < 1e-12, worst
print("✓ все распределения суммируются в 1")
print()

c_v = M.c[CTX]
print("контекст «{v}» (c = {c})".format(v=CTX, c=c_v), f"λ = {lam(M, CTX):.3f}")
rows = []
for w in sorted(M.pred, key=lambda w: (-M.big[CTX, w], -p_cont(M, w))):
    rows.append([w, M.big[CTX, w]] + [f(M, w, CTX) for f in METHODS.values()])
probs = pd.DataFrame(rows, columns=["w", "c"] + list(METHODS))
print(probs.head(6).round(3).to_string(index=False))
unseen = probs[probs["c"] == 0]
print("масса невиданных:", ", ".join(f"{name} {unseen[name].sum():.2f}" for name in METHODS))

top = probs.head(6)
x = np.arange(len(top))
plt.figure(figsize=(8, 3))
for i, name in enumerate(METHODS):
    plt.bar(x + (i - 1.5) * 0.2, top[name], width=0.2, label=name)
plt.xticks(x, top["w"])
plt.title("P(w | " + CTX + ")")
plt.legend()
plt.tight_layout()
plt.show()

### Почему продолжение, а не частота

Слово «йорк» встречается 4 раза, а «чай» 3 раза. Но «йорк» стоит только после «нью», а «чай» после трёх разных слов. В новом контексте (где ни то, ни другое не встречалось) обычный откат к частоте слова предпочтёт «йорк», а Кнезер–Ней — «чай». Обе вероятности умножаются на одно и то же $\lambda(v)$, поэтому от контекста зависит их величина, но не порядок. Сравним «абсолютное дисконтирование» (та же формула, но вместо $P_{cont}$ берётся частота слова $P(w) = c(w)/N$) с Кнезером–Неем.

In [ ]:
A, B = PAIR
rows = []
for w in (A, B):
    preds = sorted({v for (v, x) in M.big if x == w})
    rows.append([w, M.uni[w], M.n1_before[w], M.uni[w] / M.N, p_cont(M, w), ", ".join(preds)])
print(pd.DataFrame(rows, columns=["слово", "c(w)", "N1+(• w)", "P(w)", "P_cont(w)", "v"]).round(3).to_string(index=False))
print()

def abs_disc(m, w, v, d=0.75):
    # та же формула, но откат к частоте слова
    return max(m.big[v, w] - d, 0) / m.c[v] + lam(m, v, d) * m.uni[w] / m.N

for v in NEW_CTXS:
    assert M.big[v, A] == 0 and M.big[v, B] == 0
    print(f"контекст «{v}» (λ = {lam(M, v):.3f}):  откат к частоте P({A})={abs_disc(M, A, v):.4f}  P({B})={abs_disc(M, B, v):.4f}   |   "
          f"Кнезер–Ней P({A})={kn(M, A, v):.4f}  P({B})={kn(M, B, v):.4f}")

## 3. Перплексия

Перплексия на тексте из N предсказанных токенов (включая `</s>`):
$$PP = 2^{-\frac{1}{N}\sum_i \log_2 P(w_i \mid w_{i-1})}.$$

Возьмём тестовые предложения из лаборатории. Если хотя бы одна биграмма не встречалась в корпусе, $P_{MLE} = 0$, $\log_2 0 = -\infty$, и перплексия MLE бесконечна.

In [ ]:
tests = ["папа любит кофе", "мама пьёт чай", "москва большой город", "нью йорк красивый город"]

In [ ]:
def sentence_report(m, sentence, methods, show=True):
    toks = [BOS] + sentence.split() + [EOS]
    logs = {name: 0.0 for name in methods}
    rows = []
    for v, w in zip(toks, toks[1:]):
        row = [f"{v} → {w}", m.big[v, w]]
        for name, f in methods.items():
            p = f(m, w, v)
            logs[name] += math.log2(p) if p > 0 else -math.inf
            row.append(p)
        rows.append(row)
    n = len(toks) - 1
    pp = {name: 2 ** (-lp / n) for name, lp in logs.items()}
    if show:
        print(f"«{sentence}»  (N = {n})")
        print(pd.DataFrame(rows, columns=["биграмма", "c"] + list(methods)).round(3).to_string(index=False))
        print("   перплексия:", ", ".join(f"{k} = {v:.2f}" for k, v in pp.items()))
        print()
    return pp

lab_methods = {"MLE": mle, "add-1": addk, "KN d=0.75": kn}
for s in tests:
    sentence_report(M, s, lab_methods)

### Корпус побольше: подбор k и d

На 11 предложениях трудно что-то оценивать. Добавим к ним ещё 25 предложений из того же мира (чай, кофе, города, семья), а 6 новых предложений отложим для проверки. Все слова тестовых предложений есть в словаре, но некоторых биграмм нет. Подберём k и d по перплексии на отложенных предложениях. Свои тексты можно загрузить через панель «Файлы» (например, `corpus.txt`, одно предложение в строке).

In [ ]:
more = [
    "бабушка пьёт горячий чай", "дедушка пьёт крепкий кофе", "папа любит крепкий кофе", "бабушка любит зелёный чай",
    "я пью зелёный чай", "мама видела москву", "папа видел москву", "я видел большой город", "казань красивый город",
    "казань большой город", "бабушка живёт в казани", "мама живёт в москве", "я живу в москве",
    "папа живёт в нью йорке", "нью йорк шумный город", "москва шумный город", "дедушка любит тихий город",
    "бабушка любит тихий вечер", "вечером мама пьёт чай", "утром папа пьёт кофе", "утром я пью кофе",
    "вечером я пью чай", "дедушка видел казань", "мама любит горячий кофе", "я люблю красивый город",
]
heldout = ["папа любит горячий чай", "бабушка пьёт кофе", "утром мама пьёт чай", "казань шумный город",
           "дедушка живёт в москве", "я видел нью йорк"]

In [ ]:
def corpus_pp(m, sentences, f):
    logp, n = 0.0, 0
    for s in sentences:
        toks = [BOS] + s.split() + [EOS]
        for v, w in zip(toks, toks[1:]):
            p = f(m, w, v)
            if p == 0:
                return math.inf
            logp += math.log2(p)
            n += 1
    return 2 ** (-logp / n)

M2 = BigramCounts(corpus + more)
print(f"{M2.n_sents} предложений, {M2.N} токенов (без <s>), |V| (слова + </s>) = {M2.V}, {M2.n1_all} разных биграмм")
assert check_sums(M2) < 1e-12
assert all(w in M2.pred for s in heldout for w in s.split())
for s in heldout:
    toks = [BOS] + s.split() + [EOS]
    missing = [f"{v} {w}" for v, w in zip(toks, toks[1:]) if M2.big[v, w] == 0]
    print(f"   {s:<32} невиданные биграммы: {missing}")
print()
print("MLE:", corpus_pp(M2, heldout, mle))

ks = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
ds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
pp_k = [corpus_pp(M2, heldout, lambda m, w, v: addk(m, w, v, k)) for k in ks]
pp_d = [corpus_pp(M2, heldout, lambda m, w, v: kn(m, w, v, d)) for d in ds]
best_k, best_d = ks[int(np.argmin(pp_k))], ds[int(np.argmin(pp_d))]
print(f"add-k: k=1 → {pp_k[ks.index(1.0)]:.2f};  лучшее k = {best_k} → {min(pp_k):.2f}")
print(f"KN:    d=0.75 → {pp_d[ds.index(0.75)]:.2f};  лучшее d = {best_d} → {min(pp_d):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(ks, pp_k, "o-"); ax[0].set_xscale("log"); ax[0].set_xlabel("k"); ax[0].set_title("Перплексия add-k на отложенных предложениях")
ax[1].plot(ds, pp_d, "s-", color="#17907E"); ax[1].set_xlabel("d"); ax[1].set_title("Перплексия Кнезера–Нея")
for a in ax:
    a.set_ylabel("перплексия")
plt.tight_layout()
plt.show()

## 4. Генерация с температурой

Модель порождает текст: начинаем с `<s>`, берём распределение $P(w \mid v)$, случайно выбираем слово и повторяем, пока не выпадет `</s>`. **Температура** $T$ меняет распределение: $p_i^{1/T}$ с нормировкой. При $T < 1$ распределение острее (модель повторяет частое), при $T > 1$ — площе (больше неожиданных пар). Отмечаем `*` биграммы, которых не было в корпусе: MLE таких не создаёт никогда, а Кнезер–Ней отдаёт им заметную долю вероятности, поэтому уже при T = 1 часть предложений звучит странно. Даже пара `<s> → </s>` (пустое предложение) получает у Кнезера–Нея около 2 %: `</s>` стоит после многих разных слов, и его $P_{cont}$ велико.

In [ ]:
rng = np.random.default_rng(24)

def next_dist(m, v, f, T=1.0):
    p = np.array([f(m, w, v) for w in m.pred])
    p = p ** (1 / T)
    return p / p.sum()

def generate(m, f, T=1.0, max_len=12, verbose=False):
    v, out = BOS, []
    while len(out) < max_len:
        p = next_dist(m, v, f, T)
        i = rng.choice(len(m.pred), p=p)
        w = m.pred[i]
        if verbose:
            top = np.argsort(-p)[:4]
            print(f"   {v:>10} → " + ", ".join(f"{m.pred[j]} {p[j]:.2f}" for j in top) + f"   ⇒ {w}")
        if w == EOS:
            break
        out.append(w + ("" if m.big[v, w] else "*"))
        v = w
    return " ".join(out) if out else "(сразу выпал </s>: пустое предложение)"

print("KN, T = 1:")
print("   ⇒", generate(M2, kn, 1.0, verbose=True))
print()
for T in (0.3, 1.0, 2.0):
    print(f"KN, температура T = {T}")
    for _ in range(4):
        print("   ", generate(M2, kn, T))
print("MLE, температура T = 1.0")
for _ in range(4):
    print("   ", generate(M2, mle, 1.0))
print("* — биграммы нет в корпусе")

v0 = GEN_CTX
fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for a, T in zip(ax, (0.3, 1.0, 2.0)):
    p = next_dist(M2, v0, kn, T)
    top = np.argsort(-p)[:8]
    a.bar([M2.pred[j] for j in top], p[top], color="#2B55C9")
    a.set_title("T = " + str(T))
    a.tick_params(axis="x", rotation=60)
ax[0].set_ylabel("вероятность")
fig.suptitle("Распределение после «{v}»".format(v=v0))
plt.tight_layout()
plt.show()

## 5. То же в `nltk.lm`

В `nltk.lm` есть `MLE`, `Laplace`, `Lidstone` и `KneserNeyInterpolated`. Два отличия от нашего кода: `nltk` добавляет в словарь `<s>` и `<UNK>`, поэтому у Лапласа знаменатель $c(v) + |V| + 2$ (и сумма по нашим $|V|$ словам немного меньше 1), а у Кнезера–Нея скидка по умолчанию 0,1 — зададим 0,75. Сравним вероятности и перплексию с нашими. Затем обучим триграммную модель Кнезера–Нея, которую мы сами не писали.

In [ ]:
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline

train_tokens = [s.split() for s in corpus + more]

def fit(model, order=2):
    data, vocab = padded_everygram_pipeline(order, train_tokens)
    model.fit(data, vocab)
    return model

lm_mle, lm_lap, lm_kn = fit(MLE(2)), fit(Laplace(2)), fit(KneserNeyInterpolated(2, discount=0.75))
print("nltk |vocab| =", len(lm_lap.vocab), "| наш код |V| =", M2.V)

held_bigrams = [(v, w) for s in heldout for v, w in zip([BOS] + s.split(), s.split() + [EOS])]
diff_kn = max(abs(lm_kn.score(w, [v]) - kn(M2, w, v)) for v, w in held_bigrams)
lap_nltk_formula = lambda m, w, v: (m.big[v, w] + 1) / (m.c[v] + m.V + 2)
diff_lap = max(abs(lm_lap.score(w, [v]) - lap_nltk_formula(M2, w, v)) for v, w in held_bigrams)
print(f"KN: наибольшая разница вероятностей = {diff_kn:.1e}")
print(f"Laplace (c+1)/(c(v)+|V|+2): наибольшая разница вероятностей = {diff_lap:.1e}")
v0 = M2.words[0]
print(f"сумма P_Laplace по нашему словарю у nltk («{v0}»): {sum(lm_lap.score(w, [v0]) for w in M2.pred):.3f}")
print()
rows = [
    ("MLE", corpus_pp(M2, heldout, mle), lm_mle.perplexity(held_bigrams)),
    ("Laplace", corpus_pp(M2, heldout, addk), lm_lap.perplexity(held_bigrams)),
    ("KN d=0.75", corpus_pp(M2, heldout, kn), lm_kn.perplexity(held_bigrams)),
]
print(pd.DataFrame(rows, columns=["метод", "наш код", "nltk"]).round(3).to_string(index=False))

lm3 = fit(KneserNeyInterpolated(3, discount=0.75), order=3)
tri = [tuple(t[i - 2:i + 1]) for s in heldout for t in [[BOS, BOS] + s.split() + [EOS]] for i in range(2, len(t))]
print()
print(f"триграммный Кнезер–Ней (nltk), перплексия: {lm3.perplexity(tri):.2f}")
gen = lm_kn.generate(12, text_seed=[BOS], random_seed=3)
print("генерация nltk:", " ".join(gen[:gen.index(EOS)] if EOS in gen else gen))   # nltk не останавливается на </s>: обрезаем сами

## 6. Задания

1. Добавьте в корпус раздела 1 предложение «папа любит чай» и пересчитайте таблицу вероятностей для контекста «пьёт» и «папа». Почему изменилось распределение после «пьёт», хотя в новом предложении этого слова нет? (Подсказка: $P_{cont}$.)
2. Найдите контекст из большого корпуса, у которого $\lambda(v)$ (доля массы для отката у Кнезера–Нея при d = 0,75) наибольшая. Объясните через $N_{1+}(v\,\bullet)$ и $c(v)$.
3. Реализуйте **откат по Кацу без скидки** («тупой откат», stupid backoff): $S(w \mid v) = P_{MLE}(w \mid v)$, если $c(v\,w) > 0$, иначе $0{,}4 \cdot P(w)$. Убедитесь, что сумма по словарю **не** равна 1. Почему такую модель нельзя сравнивать по перплексии?
4. Сгенерируйте по 200 предложений моделью Кнезера–Нея при T = 0,5 и T = 1,5. Сравните среднюю длину предложения и долю биграмм, которых не было в корпусе.
5. Обучите в `nltk` модели Кнезера–Нея порядка 2, 3 и 4 на большом корпусе и сравните их перплексию на `heldout` (n-граммы для порядка n стройте как в разделе 5, с n − 1 токенами `<s>`). Помогает ли длинный контекст на таком маленьком корпусе? Почему?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь